# Script to replay trials and save as a silent mp4.

This notebook will generate an mp4 in `./videos/`. This video will render a
sequence of trials as the monkey sees it, except the monkey's gaze position is
rendered post-hoc as a green cross. It will also display the trial phase on top.
It is the same video as `demo_replay_trials_with_neurons.ipynb` makes, but
without the neurons (no spike raster and no audio).

By default, this notebook runs on a session included in our OSF cache. If you
would like to run this notebook on trials from other sessions, you must first
download the behavior dandi file for that session by navigating to the root
directory (`../..`) and running
```
$ python3 download_dandi_data.py --subject=Perle --session=$your_session
```


In [1]:
"""Imports."""

import ast
import sys
from moviepy import CompositeVideoClip, ImageSequenceClip
import numpy as np
from pathlib import Path
from pynwb import NWBHDF5IO
from PIL import Image, ImageDraw, ImageFont
import task_state as task_state_lib
import renderer as renderer_lib

# The repository root (where paths.py lives), found from this notebook's folder.
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'paths.py').exists())
sys.path.insert(0, str(_root))
import paths

In [2]:
"""Settings."""

BEHAVIOR_DATA_DIR = paths.BEHAVIOR_NWB
WRITE_DIR = paths.TASK_VISUALIZATION / "videos"
WRITE_DIR.mkdir(exist_ok=True, parents=True)
SUBJECT = "Perle"
SESSION = "2022-06-11"
# Completed, correct 2-object trials with steady fixation, played in this order in one video.
TRIALS = [251, 401, 349, 1156, 402, 465]


In [3]:
"""Load data."""

# Load behavior data
behavior_nwb_file_path = BEHAVIOR_DATA_DIR / f"sub-{SUBJECT}/sub-{SUBJECT}_ses-{SESSION}_behavior+task.nwb"

# Read behavior data from nwb file
with NWBHDF5IO(behavior_nwb_file_path, "r") as io:
    print(f"Processing {behavior_nwb_file_path.relative_to(paths.ROOT)}")
    nwbfile = io.read()
    all_trials_df = nwbfile.trials.to_dataframe()
    all_display_df = nwbfile.intervals['display'].to_dataframe()

# Resolve identities
all_trials_df["stimulus_object_identities"] = all_trials_df.stimulus_object_identities.apply(
    lambda x: ast.literal_eval(x))
all_trials_df["num_objects"] = all_trials_df.stimulus_object_identities.apply(lambda x: len(x))

# Filter 2-object correct trials
all_trials_df = all_trials_df[
    (all_trials_df.num_objects == 2)
    & (all_trials_df.broke_fixation == False)
    & (all_trials_df.reward_duration > 0)
]

# Reset index
all_trials_df["trial_id"] = all_trials_df.index
all_trials_df = all_trials_df.reset_index(drop=True)
trial_ids = all_trials_df.trial_id.values

Processing cache/dandi_data/behavior/sub-Perle/sub-Perle_ses-2022-06-11_behavior+task.nwb


In [4]:
"""Make video."""

# Constants
frames_between_trials = 20
_SPEED = 0.5
IMAGE_SIZE = 1024

# def _render_task_phase(phase, height=64):
def _render_task_phase(phase, height=128):
    # Render text in buffer
    # font = ImageFont.load_default(size=42)
    font = ImageFont.load_default(size=84)
    buffer = Image.new('RGBA', (IMAGE_SIZE, height), (0, 0, 0, 255))
    draw = ImageDraw.Draw(buffer)
    # capitalize phase
    text = phase.capitalize()
    # Draw text centered
    text_width = draw.textlength(text, font=font)
    text_height = font.size
    left_pad = (IMAGE_SIZE - text_width) // 2
    y = (height - text_height) // 2
    # Draw text
    draw.text((left_pad, y), text, fill=(255, 255, 255, 255), font=font)
    # Convert to numpy array
    buffer = np.array(buffer)[:, :, :3]
    return buffer

def _render_trial(trial_df, display_df):
    """Render trial."""
    # Get task state and renderer
    stim_positions = ast.literal_eval(trial_df.stimulus_object_positions.values[0])
    stim_identities = trial_df.stimulus_object_identities.values[0]
    stim_target_str = trial_df.stimulus_object_target.values[0]
    stim_target = stim_target_str[1:-1].split(", ")
    stim_target = [x == 'true' for x in stim_target]
    task_state = task_state_lib.get_task_state(
        stim_positions, stim_identities, stim_target)
    for p in task_state['prey']:
        p.opacity = 255
    task_state['eye'][0].opacity = 255
    background_indices = (0, 0)
    renderer = renderer_lib.Renderer(image_size=(IMAGE_SIZE, IMAGE_SIZE))

    # Iterate through timesteps and render frames
    frames = []
    num_timesteps = len(display_df)
    for timestep in range(num_timesteps):
        fixation_cross_scale = display_df.loc[timestep, 'fixation_cross_scale']
        eye_position = display_df.loc[timestep, 'closed_loop_eye_position']
        task_phase = display_df.loc[timestep, 'task_phase']

        # Modify state according to timestep
        task_state['fixation'][0].scale = fixation_cross_scale
        task_state['eye'][0].position = eye_position
        task_state['fw'][0].position = eye_position
        blank = False
        if task_phase == 'fixation':
            for p in task_state['prey']:
                p.opacity = 0
        elif task_phase == 'stimulus':
            for p in task_state['prey']:
                p.opacity = 255
        elif task_phase == 'delay':
            for p in task_state['prey']:
                p.opacity = 0
            blank = trial_df.delay_object_blanks.values[0]
        elif task_phase == 'cue':
            task_state['fixation'][0].opacity = 0
            for p in task_state['prey']:
                p.opacity = 0
            task_state['cue'][0].opacity = 255
            blank = trial_df.delay_object_blanks.values[0]
        elif task_phase == 'response':
            task_state['fixation'][0].opacity = 0
            for p in task_state['prey']:
                p.opacity = 0
            task_state['cue'][0].opacity = 150
            task_state['fw'][0].opacity = 255
            blank = trial_df.delay_object_blanks.values[0]
        elif task_phase == 'reveal':
            task_state['fixation'][0].opacity = 0
            for p in task_state['prey']:
                p.opacity = 145
            task_state['cue'][0].opacity = 150
            task_state['fw'][0].opacity = 255
        else:
            raise ValueError(f'Unknown task_phase: {task_phase}')
        
        image = renderer(task_state, background_indices, blank=blank)
        
        # Add task phase text
        task_phase_image = _render_task_phase(task_phase)
        image = np.concatenate((task_phase_image, image), axis=0)
        
        # Append frame
        frames.append(image)
        
    return frames

def _make_video(trials):
    """Render `trials` in order as one silent mp4 in WRITE_DIR."""
    # Get videos per trial
    video_per_trial = []
    for trial_index in trials:
        # Compute trial and display dataframes for the selected trial
        trial_df = all_trials_df[all_trials_df.trial_id == trial_index]
        display_df = all_display_df[
            (all_display_df['start_time'] > trial_df['phase_stimulus_time'].values[0] - 0.25) &
            (all_display_df['stop_time'] < trial_df['stop_time'].values[0])
        ].reset_index(drop=True)

        # Get frames
        frames = _render_trial(trial_df, display_df)

        # Add blank frames between trials
        empty_frame = np.zeros_like(frames[0], dtype=np.uint8)
        for _ in range(frames_between_trials):
            frames.append(empty_frame)

        # Make video
        video_per_trial.append(ImageSequenceClip(frames, fps=int(_SPEED * 60)))

    # Write the result
    clips = [video_per_trial[0]]
    for v in video_per_trial[1:]:
        clips.append(v.with_start(clips[-1].end))

    composite = CompositeVideoClip(clips)
    name = f"{SUBJECT}_{SESSION}_trials_{'_'.join(str(t) for t in trials)}"
    title = f"{SUBJECT} {SESSION}, trials {', '.join(str(t) for t in trials)}"
    # Relative to this folder, so MoviePy's progress lines do not print the machine's own path.
    composite.write_videofile(
        str((WRITE_DIR / f"{name}.mp4").relative_to(Path.cwd())),
        audio=False,
        ffmpeg_params=["-metadata", f"title={title}"],
    )


_make_video(TRIALS)


MoviePy - Building video videos/Perle_2022-06-11_trials_251_401_349_1156_402_465.mp4.
MoviePy - Writing video videos/Perle_2022-06-11_trials_251_401_349_1156_402_465.mp4



frame_index:   0%|          | 0/1225 [00:00<?, ?it/s, now=None]

frame_index:   1%|          | 7/1225 [00:00<00:20, 60.26it/s, now=None]

frame_index:   1%|          | 15/1225 [00:00<00:17, 67.70it/s, now=None]

frame_index:   2%|▏         | 23/1225 [00:00<00:16, 71.22it/s, now=None]

frame_index:   3%|▎         | 31/1225 [00:00<00:16, 73.39it/s, now=None]

... (151 outputs left out)

frame_index:  98%|█████████▊| 1206/1225 [00:17<00:00, 72.62it/s, now=None]

frame_index:  99%|█████████▉| 1214/1225 [00:17<00:00, 73.37it/s, now=None]

frame_index: 100%|█████████▉| 1222/1225 [00:17<00:00, 73.55it/s, now=None]

MoviePy - Done !
MoviePy - video ready videos/Perle_2022-06-11_trials_251_401_349_1156_402_465.mp4
